# Homework 7 – Final Exam
## Automating Shipment Delay and Missing Document Review

Name:Ryan Cooper  
Course:BAN 5600  
Professor: Omar El Mghari  
Date: 05/09/2026

AI Use Statement:
I used AI as a study support tool to help me understand the assignment, organize my code, and improve my explanations. I reviewed the final notebook myself, ran the code on my own, and I am responsible for the final submission and video walkthrough.

In [1]:
# Import pandas so we can work with data tables
import pandas as pd

# Import files so we can upload and download CSV files in Colab
from google.colab import files

In [2]:
# Upload the shipment CSV file from your computer
uploaded = files.upload()

# Save the uploaded file name
file_name = list(uploaded.keys())[0]

# Show the uploaded file name
print("Uploaded file name:", file_name)

Saving Shipment_Tracker - ST.csv to Shipment_Tracker - ST.csv
Uploaded file name: Shipment_Tracker - ST.csv


In [3]:
# Read the uploaded CSV file into a pandas DataFrame
df = pd.read_csv(file_name)

# Show the first 5 rows
df.head()

,Order_ID,Customer_Name,Ship_Date,Expected_Delivery,Actual_Delivery,Shipment_Status,Missing_Documents
0,1001,Alpha Stores,2026-04-01,2026-04-05,2026-04-07,Delivered,No
1,1002,Beta Supply,2026-04-02,2026-04-06,NaN,In Transit,Yes
2,1003,Central Parts,2026-04-03,2026-04-07,2026-04-07,Delivered,No
3,1004,Delta Wholesale,2026-04-04,2026-04-08,2026-04-11,Delivered,No
4,1005,Empire Goods,2026-04-05,2026-04-09,NaN,In Transit,No


In [4]:
# Convert date columns into datetime format
# This is important so Python can compare dates correctly
df["Ship_Date"] = pd.to_datetime(df["Ship_Date"], errors="coerce")
df["Expected_Delivery"] = pd.to_datetime(df["Expected_Delivery"], errors="coerce")
df["Actual_Delivery"] = pd.to_datetime(df["Actual_Delivery"], errors="coerce")

# Define today's date for checking late in-transit shipments
today_check = pd.Timestamp("2026-04-12")

# Preview the cleaned data
df.head()

,Order_ID,Customer_Name,Ship_Date,Expected_Delivery,Actual_Delivery,Shipment_Status,Missing_Documents
0,1001,Alpha Stores,2026-04-01,2026-04-05,2026-04-07,Delivered,No
1,1002,Beta Supply,2026-04-02,2026-04-06,NaT,In Transit,Yes
2,1003,Central Parts,2026-04-03,2026-04-07,2026-04-07,Delivered,No
3,1004,Delta Wholesale,2026-04-04,2026-04-08,2026-04-11,Delivered,No
4,1005,Empire Goods,2026-04-05,2026-04-09,NaT,In Transit,No


In [5]:
# Create a DataFrame for delivered shipments that arrived late
delayed_delivered = df[
    (df["Actual_Delivery"].notna()) &
    (df["Actual_Delivery"] > df["Expected_Delivery"])
].copy()

# Add an issue label
delayed_delivered["Issue_Type"] = "Delivered Late"

print("Delayed delivered shipments found:", len(delayed_delivered))
delayed_delivered

Delayed delivered shipments found: 3


,Order_ID,Customer_Name,Ship_Date,Expected_Delivery,Actual_Delivery,Shipment_Status,Missing_Documents,Issue_Type
0,1001,Alpha Stores,2026-04-01,2026-04-05,2026-04-07,Delivered,No,Delivered Late
3,1004,Delta Wholesale,2026-04-04,2026-04-08,2026-04-11,Delivered,No,Delivered Late
5,1006,Fast Retail,2026-04-06,2026-04-10,2026-04-13,Delivered,Yes,Delivered Late


In [6]:
# Create a DataFrame for shipments with missing documents
missing_docs = df[
    df["Missing_Documents"] == "Yes"
].copy()

# Add an issue label
missing_docs["Issue_Type"] = "Missing Documents"

print("Shipments with missing documents found:", len(missing_docs))
missing_docs

Shipments with missing documents found: 2


,Order_ID,Customer_Name,Ship_Date,Expected_Delivery,Actual_Delivery,Shipment_Status,Missing_Documents,Issue_Type
1,1002,Beta Supply,2026-04-02,2026-04-06,NaT,In Transit,Yes,Missing Documents
5,1006,Fast Retail,2026-04-06,2026-04-10,2026-04-13,Delivered,Yes,Missing Documents


In [7]:
# Create a DataFrame for shipments still in transit but already late
in_transit_late = df[
    (df["Shipment_Status"] == "In Transit") &
    (df["Actual_Delivery"].isna()) &
    (df["Expected_Delivery"] < today_check)
].copy()

# Add an issue label
in_transit_late["Issue_Type"] = "In Transit and Late"

print("Late in-transit shipments found:", len(in_transit_late))
in_transit_late

Late in-transit shipments found: 2


,Order_ID,Customer_Name,Ship_Date,Expected_Delivery,Actual_Delivery,Shipment_Status,Missing_Documents,Issue_Type
1,1002,Beta Supply,2026-04-02,2026-04-06,NaT,In Transit,Yes,In Transit and Late
4,1005,Empire Goods,2026-04-05,2026-04-09,NaT,In Transit,No,In Transit and Late


In [8]:
# Combine all flagged shipments into one exception report
exception_report = pd.concat(
    [delayed_delivered, missing_docs, in_transit_late],
    ignore_index=True
)

# Remove duplicate records if the same order appears more than once
exception_report = exception_report.drop_duplicates(subset=["Order_ID", "Issue_Type"]).copy()

print("Total flagged exception records:", len(exception_report))
exception_report

Total flagged exception records: 7


,Order_ID,Customer_Name,Ship_Date,Expected_Delivery,Actual_Delivery,Shipment_Status,Missing_Documents,Issue_Type
0,1001,Alpha Stores,2026-04-01,2026-04-05,2026-04-07,Delivered,No,Delivered Late
1,1004,Delta Wholesale,2026-04-04,2026-04-08,2026-04-11,Delivered,No,Delivered Late
2,1006,Fast Retail,2026-04-06,2026-04-10,2026-04-13,Delivered,Yes,Delivered Late
3,1002,Beta Supply,2026-04-02,2026-04-06,NaT,In Transit,Yes,Missing Documents
4,1006,Fast Retail,2026-04-06,2026-04-10,2026-04-13,Delivered,Yes,Missing Documents
5,1002,Beta Supply,2026-04-02,2026-04-06,NaT,In Transit,Yes,In Transit and Late
6,1005,Empire Goods,2026-04-05,2026-04-09,NaT,In Transit,No,In Transit and Late


In [9]:
# Create a DataFrame for urgent follow-up orders
# These are orders that are delayed and also missing documents
urgent_follow_up = df[
    (
        ((df["Actual_Delivery"].notna()) & (df["Actual_Delivery"] > df["Expected_Delivery"])) |
        ((df["Shipment_Status"] == "In Transit") & (df["Actual_Delivery"].isna()) & (df["Expected_Delivery"] < today_check))
    ) &
    (df["Missing_Documents"] == "Yes")
].copy()

print("Urgent follow-up orders found:", len(urgent_follow_up))
urgent_follow_up

Urgent follow-up orders found: 2


,Order_ID,Customer_Name,Ship_Date,Expected_Delivery,Actual_Delivery,Shipment_Status,Missing_Documents
1,1002,Beta Supply,2026-04-02,2026-04-06,NaT,In Transit,Yes
5,1006,Fast Retail,2026-04-06,2026-04-10,2026-04-13,Delivered,Yes


In [10]:
# Build summary lists for management review
delayed_order_ids = list(pd.concat([delayed_delivered["Order_ID"], in_transit_late["Order_ID"]]).drop_duplicates())
missing_doc_order_ids = list(missing_docs["Order_ID"].drop_duplicates())
urgent_order_ids = list(urgent_follow_up["Order_ID"].drop_duplicates())

# Print a summary of the review results
print("===== Shipment Exception Summary =====")
print("Total delayed shipments:", len(delayed_order_ids))
print("Shipments with missing documents:", len(missing_doc_order_ids))
print("Flagged exception records:", len(exception_report))
print("Delayed Order IDs:", delayed_order_ids)
print("Missing Document Order IDs:", missing_doc_order_ids)
print("Urgent Follow-Up Order IDs:", urgent_order_ids)

===== Shipment Exception Summary =====
Total delayed shipments: 5
Shipments with missing documents: 2
Flagged exception records: 7
Delayed Order IDs: [1001, 1004, 1006, 1002, 1005]
Missing Document Order IDs: [1002, 1006]
Urgent Follow-Up Order IDs: [1002, 1006]


In [11]:
# Keep only the main columns for the final report
final_report = exception_report[
    [
        "Order_ID",
        "Customer_Name",
        "Ship_Date",
        "Expected_Delivery",
        "Actual_Delivery",
        "Shipment_Status",
        "Missing_Documents",
        "Issue_Type"
    ]
].copy()

print("===== Final Shipment Exception Report =====")
final_report

===== Final Shipment Exception Report =====


,Order_ID,Customer_Name,Ship_Date,Expected_Delivery,Actual_Delivery,Shipment_Status,Missing_Documents,Issue_Type
0,1001,Alpha Stores,2026-04-01,2026-04-05,2026-04-07,Delivered,No,Delivered Late
1,1004,Delta Wholesale,2026-04-04,2026-04-08,2026-04-11,Delivered,No,Delivered Late
2,1006,Fast Retail,2026-04-06,2026-04-10,2026-04-13,Delivered,Yes,Delivered Late
3,1002,Beta Supply,2026-04-02,2026-04-06,NaT,In Transit,Yes,Missing Documents
4,1006,Fast Retail,2026-04-06,2026-04-10,2026-04-13,Delivered,Yes,Missing Documents
5,1002,Beta Supply,2026-04-02,2026-04-06,NaT,In Transit,Yes,In Transit and Late
6,1005,Empire Goods,2026-04-05,2026-04-09,NaT,In Transit,No,In Transit and Late


In [12]:
# Save the final report as a new CSV file
final_report.to_csv("shipment_exception_report.csv", index=False)

print("The shipment exception report has been saved as shipment_exception_report.csv")

The shipment exception report has been saved as shipment_exception_report.csv


In [13]:
# Download the final report to your computer
files.download("shipment_exception_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>